# Resume-Based Internship Recommendation System using RAG
## Infosys AI/ML Internship Assignment — Comprehensive Experimental Notebook

This notebook demonstrates the complete end-to-end pipeline:
1. **Resume Data Parsing & Cleaning**
2. **Internship Dataset Ingestion**
3. **Dense Embedding Generation** (`sentence-transformers/all-MiniLM-L6-v2`)
4. **Persistent Vector Database** (`ChromaDB` with Cosine Similarity)
5. **Semantic Similarity & Hybrid Search Retrieval**
6. **Multi-Factor Weighted Matching & Re-Ranking**
7. **Grounded RAG Explanation Generation**
8. **Empirical Evaluation** (Precision@5, Recall@5, MRR, Hit Rate@5, Latency)

In [ ]:
import sys
import os
import json
import pandas as pd
import numpy as np

# Add project root to sys.path
sys.path.append(os.path.abspath('..'))

from src.resume_preprocessing import preprocess_resume, get_all_skills_flat
from src.internship_processing import load_and_preprocess_internships
from src.embedding_generator import get_embedding_manager, generate_candidate_embedding
from src.vector_database import init_vector_database, get_or_create_collection
from src.internship_retrieval import search_matching_internships
from src.rag_generator import generate_rag_explanations
from src.evaluation import run_benchmark_evaluation

print('All modules imported successfully!')

## 1. Inspecting Candidate and Internship Datasets

In [ ]:
# Load candidate test dataset
with open('../data/test_candidates.json', 'r', encoding='utf-8') as f:
    candidates = json.load(f)

# Load internships dataset
with open('../data/internships.json', 'r', encoding='utf-8') as f:
    internships = json.load(f)

print(f'Total Test Candidates: {len(candidates)}')
print(f'Total Internship Postings: {len(internships)}')

# Sample internship distribution across 10 categories
df_internships = pd.DataFrame(internships)
df_internships['category'].value_counts()

## 2. Resume Preprocessing & Skill Normalization
We preprocess raw resume data: removing duplicate skills, normalizing canonical names (e.g. `python3` -> `Python`, `drf` -> `Django REST Framework`), handling nulls, and creating embedding text.

In [ ]:
sample_candidate = candidates[0]  # Aarav Sharma (Python + Django)
cleaned_cand, emb_text = preprocess_resume(sample_candidate)

print('Candidate Name:', cleaned_cand['candidate_name'])
print('Normalized Skills:', get_all_skills_flat(cleaned_cand))
print('\nEmbedding Text:\n', emb_text)

## 3. Persistent ChromaDB Vector Store & Embedding Verification

In [ ]:
collection, count = init_vector_database(
    json_filepath='../data/internships.json',
    persist_directory='../chroma_db'
)
print(f'ChromaDB Collection: {collection.name} with {count} records.')

# Verify embedding shape
cand_emb = generate_candidate_embedding(cleaned_cand)
print(f'Candidate Embedding Dimension: {len(cand_emb)}')
assert len(cand_emb) == 384, 'Expected 384-dimensional dense vector'

## 4. Semantic Matching & Multi-Factor Re-Ranking Execution
We retrieve the top-5 opportunities using the hybrid re-ranking pipeline combining dense cosine similarity, skill overlap, and multi-factor project/education weights.

In [ ]:
recommendations = search_matching_internships(
    candidate_data=cleaned_cand,
    top_k=5,
    collection=collection,
    use_hybrid=True,
    use_reranking=True
)

for r in recommendations:
    print(f"Rank #{r['rank']} | [{r['internship_id']}] {r['title']} ({r['company_name']})")
    print(f"  Category: {r['category']} | Location: {r['location']} ({r['work_mode']})")
    print(f"  Cosine Similarity: {r['similarity_score']*100:.1f}% | Overall Rank: {r['overall_ranking_score']*100:.1f}%")
    print(f"  Matched Skills: {r['explainability']['matched_required_skills']}")
    print('-'*70)

## 5. Grounded RAG Explanation Generation
We pass the retrieved candidates and scoring data to generate grounded match rationales and skill gap suggestions.

In [ ]:
explained = generate_rag_explanations(cleaned_cand, recommendations)
top_exp = explained[0]['ai_explanation']

print('Top Recommendation AI Analysis:')
print('Summary      :', top_exp.get('summary'))
print('Projects     :', top_exp.get('project_relevance'))
print('Improvements :', top_exp.get('suggested_improvements'))
print('Engine       :', top_exp.get('generation_method'))

## 6. Empirical Evaluation & Retrieval Strategy Comparison

In [ ]:
with open('../data/evaluation_results.json', 'r', encoding='utf-8') as f:
    eval_data = json.load(f)

df_eval = pd.DataFrame([
    {
        'Strategy': v['method_name'],
        'Precision@5': f"{v['precision_at_5']*100:.1f}%",
        'Recall@5': f"{v['recall_at_5']*100:.1f}%",
        'MRR': f"{v['mean_reciprocal_rank_mrr']:.4f}",
        'Hit Rate@5': f"{v['hit_rate_at_5']*100:.1f}%",
        'Avg Latency (ms)': f"{v['avg_latency_ms']:.1f} ms"
    }
    for k, v in eval_data['method_comparison'].items()
])
df_eval